In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import json
import glob
from pathlib import Path
import torch

import pandas as pd
from tqdm import tqdm

# # Add project root to path
# project_root = "/mnt/caiw/AttnLRP"
# if project_root not in sys.path:
#     sys.path.append(project_root)

In [ ]:
# pandas options to show more rows
pd.set_option('display.max_rows', 500)

In [ ]:
bench_name, traces_dir = "OLMo_IFEval", Path("/mnt/caiw/exp/olmo_exp/ifeval/without_think") 

In [ ]:
json_files = list(traces_dir.glob("*.json"))
print(f"Found {len(json_files)} json files.")

all_records = []
excluded_files = []

# Iterate over all found json files
for file_path in tqdm(json_files, desc="Processing files"):
    start_len = len(all_records)
    
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Failed to load {file_path}: {e}")
        excluded_files.append(file_path.name)
        continue
        
    # Helper to extract data
    def extract_rollouts(source_data, key, model_label):
        rollouts = source_data.get(key)
        if not rollouts:
            return

        # rollouts should be a list of top-k candidates
        for idx, item in enumerate(rollouts):
            # Try to get specific fields, robustly
            record = {
                "sample_id": data['metadata']['uuid'],
                "filename": file_path.name,
                "model": model_label,
                "contrast_token": data.get("contrast_token"),
                "rank": item.get("rank", idx), 
                "token_str": item.get("token_str"),
                "token_id": item.get("token_id"),
                "logits": item.get("logit", item.get("logits")),
                "eval_result": item.get("eval_result"),
                "completion": item.get("completion", ""),
                "l_instruction_phases": data.get("l_instruction_phases"),
                "completion_before_err": data.get("completion_before_err"),
            }
            all_records.append(record)

    # 2. Extract "topk_token_explore" -> model="current"
    extract_rollouts(data, "topk_token_explore", "current")
    
    if len(all_records) == start_len:
        excluded_files.append(file_path.name)

# 4. Create DataFrame
df = pd.DataFrame(all_records)
if not df.empty:
    df = df.sort_values(by=['sample_id', 'model', 'rank']).reset_index(drop=True)

# Display result info
print(f"Total records: {len(df)}")
if not df.empty:
    print("Unique samples:", df['sample_id'].nunique())

if excluded_files:
    print(f"\nFiles not captured in df ({len(excluded_files)}):")
    print(", ".join(sorted(excluded_files)))
else:
    print("\nAll files were captured in df.")

df.head()

In [ ]:
# ---------------------------------------------------------
# Filter 1: Current model top-1 eval_result check
# Rule: Top-1 prediction of 'current' model must be False (Failed)
# ---------------------------------------------------------

# Find top-1 for current model
current_model_df = df[df['model'] == 'current']
# Sort by rank to ensure we get the top prediction
top1_preds = current_model_df.sort_values(['sample_id', 'rank']).groupby('sample_id').head(1)

# Identify samples where eval_result is NOT False (i.e. True)
violating_samples_1 = top1_preds[top1_preds['eval_result'] != False]['sample_id'].values

print(f"Check 1: Found {len(violating_samples_1)} samples where current model top-1 eval_result != False.")
if len(violating_samples_1) > 0:
    print("Dropping these samples:", violating_samples_1)
    df = df[~df['sample_id'].isin(violating_samples_1)]

# # ---------------------------------------------------------
# # Filter 2: Completeness check
# # Rule: Samples must have both 'current' and '1.7b' models
# # ---------------------------------------------------------

# models_per_sample = df.groupby('sample_id')['model'].unique().apply(set)
# required_models = {'current', '1.7b'}
# violating_samples_2 = models_per_sample[models_per_sample != required_models].index.tolist()

# print(f"\nCheck 2: Found {len(violating_samples_2)} samples missing required models {required_models}.")
# if len(violating_samples_2) > 0:
#     print("Dropping these samples:", violating_samples_2)
#     df = df[~df['sample_id'].isin(violating_samples_2)]

# ---------------------------------------------------------
# Filter 3: Contrast token check
# Rule: 'contrast_token' must not be NA
# ---------------------------------------------------------

samples_missing_contrast = df[df['contrast_token'].isna()]['sample_id'].unique()

print(f"\nCheck 3: Found {len(samples_missing_contrast)} samples with missing contrast_token.")
if len(samples_missing_contrast) > 0:
    print("Dropping these samples:", samples_missing_contrast)
    df = df[~df['sample_id'].isin(samples_missing_contrast)]


# Reset index after filtering
df = df.sort_values(by=['sample_id', 'model', 'rank']).reset_index(drop=True)

print(f"\nFinal cleaned dataset: {len(df)} records, {df['sample_id'].nunique()} unique samples.")

In [ ]:
# Create a simplified DataFrame with one row per sample
# Columns: sample_id, target_token, contrast_token

# 1. Get rank-0 rows for current model to get target_token (token_str)
target_df = df[(df['model'] == 'current') & (df['rank'] == 0)][['sample_id', 'token_str', 'l_instruction_phases', 'completion_before_err']].rename(columns={'token_str': 'target_token'})

# 2. Get contrast_token (it's consistent per sample, so we drop duplicates)
contrast_info_df = df[['sample_id', 'contrast_token']].drop_duplicates()

# 3. Merge to form the requested dataframe
sample_tokens_df = pd.merge(target_df, contrast_info_df, on='sample_id', how='inner')

print(f"Derived sample_tokens_df: {len(sample_tokens_df)} rows")
sample_tokens_df.head()

In [ ]:
from attnlrp_circuit.backend.models.manager import ModelManager
from attnlrp_circuit.backend.core import AttributionEngine
import torch
import torch.nn.functional as F

# Initialize Backend
print("Initializing ModelManager...")
model_manager = ModelManager()
attribution_engine = AttributionEngine(model_manager)


# Define Models to Evaluate
# Note: Adjust paths if your local checkout/cache differs. 
# We assume "current" is the model in the trace (typically 0.6B)
# referencing the logic in previous cells/context.
ref_models = {
    # Think-SFT
    "think_sft_1000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step1000"
    },
    "think_sft_2000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step2000"
    },
    "think_sft_4000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step4000"
    },
    "think_sft_6000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step6000"
    },
    "think_sft_8000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step8000"
    },
    "think_sft_10000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step10000"
    },
    "think_sft_20000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step20000"
    },
    # "think_sft_30000": {
    #     "model_path": "allenai/Olmo-3-7B-Think-SFT",
    #     "revision": "step30000"
    # },
    "think_sft_43000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step43000"
    },
    # DPO
    "think_dpo": {
        "model_path": "allenai/Olmo-3-7B-Think-DPO",
        "revision": None
    },
    # Think
    "think_500": {
        "model_path": "allenai/Olmo-3-7B-Think",
        "revision": "step_0500"
    },
    "think_1000": {
        "model_path": "allenai/Olmo-3-7B-Think",
        "revision": "step_1000"
    },
    "think_1375": {
        "model_path": "allenai/Olmo-3-7B-Think",
        "revision": "step_1375"
    }
}

# Mapping sample_id to filename for prompt loading
if 'sample_to_filename' not in locals():
    sample_to_filename = df[['sample_id', 'filename']].drop_duplicates().set_index('sample_id')['filename'].to_dict()

results_cols = [
    "sample_id", "target_token", "contrast_token", "model",
    "target_logit", "contrast_logit", 
    "target_logit_rank", "contrast_logit_rank"
]
results_data = []

current_loaded = None

for model_label, model_info in ref_models.items():
    # Handle both string (old style) and dict (new style)
    if isinstance(model_info, dict):
        m_path = model_info['model_path']
        m_rev = model_info.get('revision')
    else:
        m_path = model_info
        m_rev = None

    print(f"\n--- Processing Model: {model_label} ({m_path}, rev={m_rev}) ---")
    
    # Load Model
    target_state = (m_path, m_rev)
    if current_loaded != target_state:
        try:
            print(f"Loading {m_path} (rev={m_rev}) in bfloat16...")
            model_manager.load_model(model_path=m_path, dtype="bfloat16", revision=m_rev)
            current_loaded = target_state
        except Exception as e:
            print(f"Failed to load {m_path}: {e}")
            continue
            
    tokenizer = model_manager.tokenizer
    
    for idx, row in tqdm(sample_tokens_df.iterrows(), total=len(sample_tokens_df), desc=f"Eval {model_label}"):
        sid = row['sample_id']
        target_token_str = row['target_token']
        contrast_token_str = row['contrast_token']
        
        # 1. Get Prompt
        filename = sample_to_filename.get(sid)
        if not filename: continue
        
        try:
            with open(traces_dir / filename, 'r') as f:
                t_data = json.load(f)
            full_prompt = t_data.get('prompt', '') + t_data.get('completion_before_err', '')
        except:
            continue

        # 2. Determine Token IDs
        # Users noted that tokens might be wrapped in quotes (e.g. "'token'"). Remove them.
        if isinstance(target_token_str, str):
            target_token_str = target_token_str.strip("'")
        if isinstance(contrast_token_str, str):
            contrast_token_str = contrast_token_str.strip("'")

        # Recompute IDs using tokenizer for both target and contrast
        try:
            t_ids = tokenizer.encode(target_token_str, add_special_tokens=False)
            target_id = t_ids[-1] if len(t_ids) > 0 else -1
            
            c_ids = tokenizer.encode(contrast_token_str, add_special_tokens=False)
            contrast_id = c_ids[-1] if len(c_ids) > 0 else -1
        except:
            target_id = -1
            contrast_id = -1
            
        if target_id == -1 or contrast_id == -1:
            continue

        # 3. Compute Logits
        try:
            with torch.no_grad():
                # compute_logits returns: topk_list, logits_tensor, input_ids
                _, logits, _ = attribution_engine.compute_logits(full_prompt)
                logits = logits.cpu().float() # Move to CPU for analysis
                
            # 4. Extract Metrics
            t_logit = logits[target_id].item()
            c_logit = logits[contrast_id].item()
            
            # Ranks (0-based)
            # Count how many logits are strictly greater than the target logit
            t_rank = (logits > t_logit).sum().item()
            c_rank = (logits > c_logit).sum().item()
            
            results_data.append({
                "sample_id": sid,
                "target_token": target_token_str,
                "contrast_token": contrast_token_str,
                "model": model_label,
                "target_logit": t_logit,
                "contrast_logit": c_logit,
                "target_logit_rank": t_rank,
                "contrast_logit_rank": c_rank
            })
            
        except Exception as e:
            print(f"Error {sid}: {e}")
            continue

# Create final DataFrame
logit_rank_df = pd.DataFrame(results_data)
print(f"Created logit_rank_df with {len(logit_rank_df)} rows.")
logit_rank_df['logit_diff'] = logit_rank_df['target_logit'] - logit_rank_df['contrast_logit']
logit_rank_df.head()

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Use the order from ref_models dictionary keys
model_order = list(ref_models.keys())

# Pivot: Index=sample_id, Columns=model, Values=logit_diff
pivot_df = logit_rank_df.pivot(index='sample_id', columns='model', values='logit_diff')

# Reindex columns using the specific order, filtering for existence
ordered_cols = [m for m in model_order if m in pivot_df.columns]
plot_df = pivot_df[ordered_cols]

fig = go.Figure()

# Add a trace for each sample. 
# We use Scattergl which renders using WebGL, allowing for many more lines with good performance.
for sample_id in plot_df.index:
    row = plot_df.loc[sample_id]
    fig.add_trace(go.Scattergl(
        x=ordered_cols,
        y=row.values,
        mode='lines',
        line=dict(color='rgba(31, 119, 180, 0.15)', width=1),
        name=str(sample_id),
        showlegend=False,
        hovertemplate=f"<b>Sample: {sample_id}</b><br>Model: %{{x}}<br>Diff: %{{y:.3f}}<extra></extra>"
    ))

# Add a mean trend line
mean_trend = plot_df.mean()
fig.add_trace(go.Scatter(
    x=mean_trend.index,
    y=mean_trend.values,
    mode='lines+markers',
    line=dict(color='red', width=4),
    name='Average',
    hovertemplate="<b>Average</b><br>Model: %{x}<br>Diff: %{y:.3f}<extra></extra>"
))

# Add zero reference line
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)

fig.update_layout(
    title=f'Logit Difference Trajectory per Sample (N={len(plot_df)})',
    xaxis_title='Model Variant',
    yaxis_title='Logit Difference (Target - Contrast)',
    template='plotly_white',
    hovermode='closest',
    height=600
)

fig.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use the order from ref_models dictionary keys
model_order = list(ref_models.keys())

# Pivot: Index=sample_id, Columns=model, Values=logit_diff
pivot_df = logit_rank_df.pivot(index='sample_id', columns='model', values='logit_diff')

# Reindex columns to match the specific order in ref_models
# We filter to ensure we only look for columns that actually exist in the dataframe
ordered_cols = [m for m in model_order if m in pivot_df.columns]
pivot_df = pivot_df.reindex(columns=ordered_cols)

plt.figure(figsize=(14, 7))

# Plot all samples (transposed so X-axis = models)
# We use a low alpha to visualize density/trends
# pivot_df.T has models as index (x-axis) and sample_ids as columns (lines)
plt.plot(pivot_df.T, color='tab:blue', alpha=0.15, linewidth=1, label='_nolegend_')

# Add a mean trend line
mean_trend = pivot_df.mean()
plt.plot(mean_trend.index, mean_trend.values, color='red', linewidth=2.5, marker='o', label='Average')

plt.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
plt.title(f'Logit Difference Trajectory per Sample (N={len(pivot_df)})')
plt.xlabel('Model Variant')
plt.ylabel('Logit Difference (Target - Contrast)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, axis='both', alpha=0.2)
plt.legend(loc='best') # Should only show 'Average' as others have label starting with _

plt.tight_layout()
plt.show()

In [ ]:
import shutil
import numpy as np
import json

# ---------------------------------------------------------
# 1. Setup Output Directory
# ---------------------------------------------------------
# New base path: exp/without_thinking/heatmaps
base_output_dir = Path(f"/mnt/caiw_repos/AttnLRP/exp/without_thinking/heatmaps/{bench_name}")
base_output_dir.mkdir(parents=True, exist_ok=True)
print(f"Heatmap Base Directory: {base_output_dir}")

# ---------------------------------------------------------
# Define Models Config (Moved up for Manifest usage)
# ---------------------------------------------------------
models_config = {
    # Think-SFT
    "think_sft_1000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step1000"
    },
    "think_sft_2000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step2000"
    },
    "think_sft_4000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step4000"
    },
    "think_sft_6000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step6000"
    },
    "think_sft_8000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step8000"
    },
    "think_sft_10000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step10000"
    },
    "think_sft_20000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step20000"
    },
    # "think_sft_30000": {
    #     "model_path": "allenai/Olmo-3-7B-Think-SFT",
    #     "revision": "step30000"
    # },
    "think_sft_43000": {
        "model_path": "allenai/Olmo-3-7B-Think-SFT",
        "revision": "step43000"
    },
    # DPO
    "think_dpo": {
        "model_path": "allenai/Olmo-3-7B-Think-DPO",
        "revision": None
    },
    # Think
    "think_500": {
        "model_path": "allenai/Olmo-3-7B-Think",
        "revision": "step_0500"
    },
    "think_1000": {
        "model_path": "allenai/Olmo-3-7B-Think",
        "revision": "step_1000"
    },
    "think_1375": {
        "model_path": "allenai/Olmo-3-7B-Think",
        "revision": "step_1375"
    }
}


# ---------------------------------------------------------
# 2. Identify Target Samples
# ---------------------------------------------------------
# NO Filtering: Process ALL samples
pivot_df = logit_rank_df.pivot(index='sample_id', columns='model', values='logit_diff')

target_sids = pivot_df.index.tolist()
print(f"Generating heatmaps for {len(target_sids)} samples (ALL)...")

# ---------------------------------------------------------
# 3. Prepare Manifest
# ---------------------------------------------------------
manifest_entries = {}

# We need token strings and IDs.
# Access logit_rank_df to get strings. mapping sid -> tokens
token_map = logit_rank_df[['sample_id', 'target_token', 'contrast_token']].drop_duplicates('sample_id').set_index('sample_id')

for sid in target_sids:
    row = pivot_df.loc[sid]
    # Check if we have token info
    if sid not in token_map.index:
        continue

    tokens = token_map.loc[sid]
    
    # Use precise model names for keys in 'logit_diffs'
    diffs = {}
    for m_label in models_config.keys():
        if pd.notna(row.get(m_label)):
            diffs[m_label] = row[m_label]
    
    manifest_entries[sid] = {
        "sample_id": sid,
        "original_id": sid,
        "filename": sample_to_filename.get(sid, f"{sid}.json"),
        "ref_token": tokens['target_token'],
        "contrast_token": tokens['contrast_token'],
        "logit_diffs": diffs
    }

# ---------------------------------------------------------
# 4. Processing Loop
# ---------------------------------------------------------

current_loaded = None # changed from current_loaded_model_path for tuple support

for model_label, config in models_config.items():
    print(f"\n--- Processing Model: {model_label} ---")
    
    model_path = config['model_path']
    revision = config.get('revision')
    
    model_output_dir = base_output_dir / model_label
    model_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Load Model
    target_state = (model_path, revision)
    if current_loaded != target_state:
        try:
            print(f"Loading {model_path} (rev={revision})...")
            model_manager.load_model(model_path=model_path, dtype="bfloat16", lrp_rule="Attn-LRP", revision=revision)
            current_loaded = target_state
        except Exception as e:
            print(f"Failed to load {model_path}: {e}")
            continue
            
    tokenizer = model_manager.tokenizer

    # Pre-calculate processing info
    processing_list = []
    
    for sid in target_sids:
        # Get Prompt
        filename = sample_to_filename.get(sid)
        if not filename: continue
        
        try:
            with open(traces_dir / filename, 'r') as f:
                t_data = json.load(f)
            full_prompt = t_data.get('prompt', '') + t_data.get('completion_before_err', '')
        except:
            continue
        
        if sid not in manifest_entries:
            continue

        # Get Token IDs (re-encode to be safe and consistent with previous step)
        t_str = manifest_entries[sid]['ref_token']
        c_str = manifest_entries[sid]['contrast_token']
        
        # Strip quotes if present (as before)
        if isinstance(t_str, str): t_str = t_str.strip("'")
        if isinstance(c_str, str): c_str = c_str.strip("'")
            
        try:
            t_ids = tokenizer.encode(t_str, add_special_tokens=False)
            target_id = t_ids[-1] if len(t_ids) > 0 else -1
            
            c_ids = tokenizer.encode(c_str, add_special_tokens=False)
            contrast_id = c_ids[-1] if len(c_ids) > 0 else -1
        except:
            continue
            
        if target_id == -1 or contrast_id == -1:
            continue
            
        processing_list.append({
            "sid": sid,
            "save_name": sid,
            "prompt": full_prompt,
            "target_id": target_id,
            "contrast_id": contrast_id
        })

    # Compute and Save
    for item in tqdm(processing_list, desc=f"Attributing {model_label}"):
        out_file = model_output_dir / f"{item['save_name']}.json"
        
        try:
            # 1. Logits
            _, _, input_tokens = attribution_engine.compute_logits(item['prompt'])
            
            # 2. Attribution
            bp_config = {
                "mode": "logit_diff",
                "strategy": "by_ref_token",
                "target_token_id": item['target_id'],
                "ref_token_id": item['contrast_id'],
                "lrp_rule": "Attn-LRP"
            }
            
            relevance = attribution_engine.compute_input_attribution(bp_config)
            
            # 3. Save
            result = {
                "meta": {
                    "sample_id": item['save_name'],
                    "model": model_label,
                    "target_token_id": item['target_id'],
                    "contrast_token_id": item['contrast_id']
                },
                "tokens": input_tokens,
                "relevance": relevance
            }
             
            # Helper to clean numpy types
            def clean(obj):
                if isinstance(obj, list): return [clean(x) for x in obj]
                if isinstance(obj, dict): return {k: clean(v) for k, v in obj.items()}
                val = obj
                if hasattr(obj, 'item'): val = obj.item()
                if isinstance(val, (float, np.float32, np.float16, np.float64)):
                    if np.isnan(val) or np.isinf(val): return 0.0
                    return float(val)
                return val
            
            with open(out_file, 'w') as f:
                json.dump(clean(result), f, indent=2)
                
        except Exception as e:
            print(f"Error {item['sid']}: {e}")

# ---------------------------------------------------------
# 5. Save Manifest
# ---------------------------------------------------------
manifest_path = base_output_dir / "manifest.json"
manifest_data = {
    "models": list(models_config.keys()),
    "samples": list(manifest_entries.values())
}

def clean_manifest(obj):
    if isinstance(obj, list): return [clean_manifest(x) for x in obj]
    if isinstance(obj, dict): return {k: clean_manifest(v) for k, v in obj.items()}
    val = obj
    if hasattr(obj, 'item'): val = obj.item()
    if isinstance(val, (float, np.float32, np.float16, np.float64)):
        if np.isnan(val) or np.isinf(val): return None
        return float(val)
    return val

with open(manifest_path, 'w') as f:
    json.dump(clean_manifest(manifest_data), f, indent=2)

## Breakdown by prompt sections

In [ ]:
import numpy as np
import json

def get_max_abs_norm(relevance, tokens, hide_first=True, hide_special=True):
    """
    Computes the normalization denominator using Max Abs strategy, 
    optionally identifying tokens to exclude from the scale calculation.
    """
    special_tokens = {'<|im_start|>', '<|im_end|>', '<think>', '</think>'}
    
    # Collect values for denominator calculation
    valid_abs_scores = []
    
    for idx, (score, token) in enumerate(zip(relevance, tokens)):
        # Skip the first token (BOS) if requested
        if hide_first and idx == 0:
            continue
            
        # Skip special tokens if requested
        if hide_special and token.strip() in special_tokens:
            continue
            
        valid_abs_scores.append(abs(score))
        
    if not valid_abs_scores:
        return 1.0
        
    # Standard Max Abs strategy
    denominator = max(valid_abs_scores)
    return denominator if denominator > 1e-9 else 1.0

def normalize_relevance(relevance, tokens, hide_first=True, hide_special=True):
    """
    Returns normalized relevance scores.
    """
    denominator = get_max_abs_norm(relevance, tokens, hide_first, hide_special)
    return [x / denominator for x in relevance]

# --- Batch Normalization ---
target_models = [
    "think_sft_1000", "think_sft_2000", "think_sft_4000", "think_sft_6000", "think_sft_8000",
    "think_sft_10000", "think_sft_20000", "think_sft_43000",
    "think_dpo",
    "think_500", "think_1000", "think_1375"
]
base_output_dir = Path("/mnt/caiw_repos/AttnLRP/exp/without_thinking/heatmaps/OLMo_IFEval")
normalized_results = {} # Primary key: sid -> Secondary key: model -> data

# Load manifest for logit_diff lookup
manifest_path = base_output_dir / "manifest.json"
manifest_lookup = {}
if manifest_path.exists():
    try:
        with open(manifest_path, 'r') as mf:
            m_data = json.load(mf)
        # Convert list to dict for faster lookup
        for entry in m_data.get("samples", []):
            manifest_lookup[str(entry["sample_id"])] = entry
        print(f"Loaded {len(manifest_lookup)} entries from manifest.")
    except Exception as e:
        print(f"Error loading manifest: {e}")
else:
    print(f"Manifest not found at {manifest_path}")


for target_model in target_models:
    model_results_dir = base_output_dir / target_model
    
    print(f"\nProcessing {target_model} from {model_results_dir} ...")

    if model_results_dir.exists():
        # Identify all JSON files in the directory
        all_json_files = list(model_results_dir.glob("*.json"))
        print(f"Found {len(all_json_files)} files in {model_results_dir}")
        
        for json_path in tqdm(all_json_files, desc=f"Normalizing {target_model}"):
            sid = json_path.stem
            # Skip non-sample files if any (e.g. manifest.json although it is usually at base, check just in case)
            if sid == "manifest":
                continue
                
            try:
                with open(json_path, 'r') as f:
                    data = json.load(f)
                
                # Defensive check for expected keys
                if 'tokens' not in data or 'relevance' not in data:
                    continue
                    
                tokens = data['tokens']
                relevance = data['relevance']
                
                # Normalize
                norm_scores = normalize_relevance(relevance, tokens, hide_first=True, hide_special=True)
                
                # Lookup logit_diff
                logit_diff = None
                if sid in manifest_lookup:
                    diffs = manifest_lookup[sid].get("logit_diffs", {})
                    logit_diff = diffs.get(target_model)
                
                # Structure: normalized_results[sid][model] = { ... }
                if sid not in normalized_results:
                    normalized_results[sid] = {}
                    
                normalized_results[sid][target_model] = {
                    "tokens": tokens,
                    "raw_relevance": relevance,
                    "norm_relevance": norm_scores,
                    "logit_diff": logit_diff
                }
            except Exception as e:
                print(f"Error processing {sid} for {target_model}: {e}")
                
    else:
        print(f"Directory {model_results_dir} not found. Skipping.")
            
print(f"\nSuccessfully processed {len(normalized_results)} samples across models.")

# Check if we have results to display a summary
if len(normalized_results) > 0:
    # Show stats for the first processed sample and the first model found for it
    sample_sid = next(iter(normalized_results))
    first_model = next(iter(normalized_results[sample_sid]))
    sample_data = normalized_results[sample_sid][first_model]
    
    print(f"\nSample Summary ({sample_sid} - {first_model}):")
    print(f"Original Range: {min(sample_data['raw_relevance']):.4f} to {max(sample_data['raw_relevance']):.4f}")
    print(f"Normalized Range: {min(sample_data['norm_relevance']):.4f} to {max(sample_data['norm_relevance']):.4f}")
    print(f"Logit Diff: {sample_data['logit_diff']}")

In [ ]:
# Function to calculate phrase relevance
def find_phrase_relevance(tokens, relevance, phrases):
    """
    Calculates total relevance for tokens corresponding to specific text phrases.
    """
    if not phrases:
        return {}

    full_text = ""
    token_spans = []
    
    for i, t in enumerate(tokens):
        start = len(full_text)
        full_text += t
        end = len(full_text)
        token_spans.append((start, end, i))
        
    results = {}
    
    for phrase in phrases:
        if not phrase: continue
        
        # Try exact match first
        start_idx = full_text.find(phrase)
        
        # If not found, try stripped or lenient match (optional improvement)
        if start_idx == -1:
            results[phrase] = {"found": False, "total_relevance": 0.0, "tokens": [], "indices": []}
            continue
            
        end_idx = start_idx + len(phrase)
        
        matched_indices = []
        matched_tokens = []
        phase_relevance = 0.0
        
        for t_start, t_end, t_idx in token_spans:
            # Check overlap
            overlap_start = max(start_idx, t_start)
            overlap_end = min(end_idx, t_end)
            
            if overlap_start < overlap_end:
                matched_indices.append(t_idx)
                matched_tokens.append(tokens[t_idx])
                phase_relevance += relevance[t_idx]
                
        results[phrase] = {
            "found": True,
            "total_relevance": phase_relevance,
            "tokens": matched_tokens,
            "indices": matched_indices
        }
        
    return results

# Process all samples in normalized_results
results_data = []
special_tokens = {'<|im_start|>', '<|im_end|>', '<think>', '</think>'}

# Create a lookup for metadata from sample_tokens_df
meta_lookup = {}
if 'sample_tokens_df' in locals():
    # Handle potential duplicates by taking the first occurrence for each sample_id
    temp_df = sample_tokens_df.drop_duplicates(subset=['sample_id']).copy()
    
    # Ensure sample_id is string to match JSON filenames
    temp_df['sample_id'] = temp_df['sample_id'].astype(str)
    
    # Check key columns
    expected_cols = ['l_instruction_phases', 'completion_before_err', 'target_token']
    available_cols = [c for c in expected_cols if c in temp_df.columns]
    
    meta_lookup = temp_df.set_index('sample_id')[available_cols].to_dict('index')
    
    if 'target_token' not in available_cols:
        print("Warning: 'target_token' column missing in sample_tokens_df.")
else:
    print("Error: sample_tokens_df not found in environment.")

# Debug: Check lookup for first key in normalized_results
if normalized_results:
    first_sid = next(iter(normalized_results))
    if first_sid in meta_lookup:
        print(f"Debug: Metadata found for {first_sid}. Keys: {list(meta_lookup[first_sid].keys())}")

# Iterate through all samples AND all models
for sid, model_data in tqdm(normalized_results.items(), desc="Aggregating Relevance"):
    for model_name, data in model_data.items():
        tokens = data['tokens']
        norm_scores = data['norm_relevance']
        logit_diff = data.get('logit_diff') # Extract logit_diff
        
        # Retrieve Metadata
        meta = meta_lookup.get(str(sid), {}) 
        l_instruction_phases = meta.get('l_instruction_phases', [])
        if not isinstance(l_instruction_phases, list):
            l_instruction_phases = [] 
            
        completion_before_err = meta.get('completion_before_err', "")
        if not isinstance(completion_before_err, str):
            completion_before_err = ""
            
        target_token = meta.get('target_token', "")
        if isinstance(target_token, str):
            target_token = target_token.strip("'")
        else:
            # If target_token is missing or not a string, we can't do the split correctly
            target_token = ""

        # 1. Analyze Instructions
        inst_results = find_phrase_relevance(tokens, norm_scores, l_instruction_phases)
        consumed_indices = set()
        total_instruction_relevance = 0.0
        inst_rel_details = []
        
        # Group Logic variables
        group1_rel = 0.0 # With target token
        group2_rel = 0.0 # Without target token
        group1_exists = False
        group2_exists = False
        
        for phrase, res in inst_results.items():
            rel_value = res['total_relevance'] if res['found'] else 0.0
            
            if res['found']:
                consumed_indices.update(res['indices'])
                total_instruction_relevance += rel_value
                
            inst_rel_details.append((phrase, rel_value))
            
            # Check if phrase belongs to Group 1 (contains target_token)
            if target_token and target_token.strip() in phrase:
                group1_rel += rel_value
                group1_exists = True
            else:
                group2_rel += rel_value
                group2_exists = True
                
        # 2. Analyze Completion (Context)
        completion_indices = set()
        completion_relevance = 0.0
        if completion_before_err:
            comp_res = find_phrase_relevance(tokens, norm_scores, [completion_before_err])
            res = comp_res.get(completion_before_err)
            if res and res['found']:
                # Avoid overlap with instructions
                new_indices = set(res['indices']) - consumed_indices
                completion_indices.update(new_indices)
                completion_relevance = sum(norm_scores[i] for i in new_indices if i < len(norm_scores))

        # 3. Analyze Special Tokens
        special_indices = set()
        for idx, token in enumerate(tokens):
            if idx == 0 or token.strip() in special_tokens:
                special_indices.add(idx)
                
        special_relevance = sum(norm_scores[i] for i in special_indices if i < len(norm_scores))

        # 4. Query (Other) Relevance
        all_indices = set(range(len(tokens)))
        other_indices = all_indices - consumed_indices - completion_indices - special_indices
        query_relevance = sum(norm_scores[i] for i in other_indices if i < len(norm_scores))
        
        sum_inst_comp_query = total_instruction_relevance + completion_relevance + query_relevance
        
        # --- New Derived Columns Logic ---
        
        # 1. is_AG_required
        # Condition: sum_inst_comp_query < 0 AND logit_diff > 1
        # Handle None logit_diff safely
        ld_val = logit_diff if logit_diff is not None else -999.0
        is_AG_required = 1 if (sum_inst_comp_query < 0 and ld_val > 1) else 0
        
        # 2. IF_cat
        if_cat = "neutral"
        
        if total_instruction_relevance < 0:
            if sum_inst_comp_query > 0:
                if_cat = "underweight_rel_inst"
            else:
                if_cat = "IF_seems_ok"
        elif total_instruction_relevance > 0:
            # Check for sub-category: underweight_rel_inst_AND_overweight_irrel_inst
            # Condition: group1 > 0 AND (group2 < 0 OR group2 not exist)
            
            cond_g1 = group1_exists and (group1_rel > 0)
            cond_g2 = (not group2_exists) or (group2_rel < 0)
            
            if cond_g1 and cond_g2:
                if_cat = "underweight_rel_inst_AND_overweight_irrel_inst"
            else:
                if_cat = "underweight_rel_inst"
                
        
        results_data.append({
            "sample_id": sid,
            "model": model_name, # Added model variant
            "logit_diff": logit_diff,
            "is_AG_required": is_AG_required,
            "IF_cat": if_cat,
            "total_instruction_relevance": total_instruction_relevance,
            "group1_instruction_relevance": group1_rel if group1_exists else None, # "na" if group 1 empty
            "group2_instruction_relevance": group2_rel if group2_exists else None,
            "completion_relevance": completion_relevance,
            "query_relevance": query_relevance,
            "special_relevance": special_relevance,
            "sum_inst_comp_query": sum_inst_comp_query,
            "inst_rel_details": inst_rel_details
        })

# Create Final DataFrame
final_df = pd.DataFrame(results_data)
print(f"Created final_df with {len(final_df)} rows.")
if not final_df.empty:
    print("Mean Instruction Relevance:", final_df['total_instruction_relevance'].mean())
    print("IF_cat distribution:\n", final_df['IF_cat'].value_counts())
    print("\nModels present:", final_df['model'].unique())
    display(final_df.head())
else:
    print("DataFrame is empty.")

In [ ]:
import matplotlib.pyplot as plt

# Metrics relevant for analysis
metrics = [
    "sum_inst_comp_query", 
    "total_instruction_relevance", 
    "group1_instruction_relevance", 
    # "group2_instruction_relevance", 
    "query_relevance", 
    "completion_relevance"
]

# Define desired order
model_order = [
    "think_sft_1000", "think_sft_2000", "think_sft_4000", "think_sft_6000", "think_sft_8000",
    "think_sft_10000", "think_sft_20000", "think_sft_43000",
    "think_dpo",
    "think_500", "think_1000", "think_1375"
]

# Calculate averages per model
# groupby 'model' and take mean of selected metrics
agg_df = final_df.groupby('model')[metrics].mean()

# Reorder rows based on model_order (filter to existing only to avoid errors if some are missing)
existing_order = [m for m in model_order if m in agg_df.index]
agg_df = agg_df.reindex(existing_order)

# Transpose for plotting: 
# X-axis: Metrics
# Bars (Hue): Models
plot_df = agg_df.T

# Plot Grouped Bar Chart (stack=False is default for kind='bar' in pandas if dataframe is used directly this way)
ax = plot_df.plot(kind='bar', figsize=(14, 7), width=0.8, alpha=0.85)

plt.title("Comparison of Average Relevance Metrics across Models")
plt.ylabel("Average Normalized Relevance")
plt.xlabel("Relevance Dimension")
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.grid(axis='y', linestyle=':', alpha=0.4)
plt.legend(title="Model", loc='upper left', bbox_to_anchor=(1.0, 1.0)) # Move legend out if many models
plt.xticks(rotation=45, ha='right')

# Add value labels
for c in ax.containers:
    ax.bar_label(c, fmt='%.3f', label_type='edge', padding=3, fontsize=8)

plt.tight_layout()
plt.show()

# Display the values
print("Average Values:")
display(agg_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define desired order
model_order = [
    "think_sft_1000", "think_sft_2000", "think_sft_4000", "think_sft_6000", "think_sft_8000",
    "think_sft_10000", "think_sft_20000", "think_sft_43000",
    "think_dpo",
    "think_500", "think_1000", "think_1375"
]

fig, axes = plt.subplots(2, 1, figsize=(14, 14))

# --- Plot 1: is_AG_required Rate ---
# Calculate mean (rate) of is_AG_required per model
ag_rate = final_df.groupby('model')['is_AG_required'].mean().reset_index()

# Filter ag_rate to only include models in our list (optional but good for consistency)
ag_rate = ag_rate[ag_rate['model'].isin(model_order)]

sns.barplot(
    data=ag_rate, 
    x='model', 
    y='is_AG_required', 
    order=model_order, # Enforce order on x-axis
    palette='viridis', 
    ax=axes[0],
    edgecolor='black', alpha=0.8
)

axes[0].set_title("Rate of 'is_AG_required' per Model\n(logit_diff > 1 AND sum_relevance < 0)")
axes[0].set_ylabel("Proportion of Samples")
axes[0].set_ylim(0, 1.05) # Proportion is 0-1
axes[0].grid(axis='y', linestyle=':', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45) 

for c in axes[0].containers:
    axes[0].bar_label(c, fmt='%.3f', padding=3)


# --- Plot 2: IF_cat Distribution ---
# Count plot normalized by model (showing percentages within each model)
# We can use update the dataframe to use group-wise normalization manually or use histplot with multiple="dodge" and stat="percent"

# Strategy: Calculate counts per model & category, then normalize
cat_counts = final_df.groupby(['model', 'IF_cat']).size().reset_index(name='count')
# Calculate total per model
model_totals = final_df.groupby('model').size().reset_index(name='total')
cat_counts = cat_counts.merge(model_totals, on='model')
cat_counts['percentage'] = cat_counts['count'] / cat_counts['total'] * 100

# Filter cat_counts
cat_counts = cat_counts[cat_counts['model'].isin(model_order)]

sns.barplot(
    data=cat_counts, 
    x='IF_cat', 
    y='percentage', 
    hue='model',
    hue_order=model_order, # Enforce order in legend/hue
    palette='Set2', 
    ax=axes[1],
    edgecolor='black', alpha=0.8
)

axes[1].set_title("Distribution of IF Categories per Model")
axes[1].set_ylabel("Percentage (%)")
axes[1].set_xlabel("IF Category")
axes[1].grid(axis='y', linestyle=':', alpha=0.3)
axes[1].legend(title="Model", bbox_to_anchor=(1.05, 1), loc='upper left')

# Rotate labels if they are long
axes[1].tick_params(axis='x', rotation=30)
for c in axes[1].containers:
    axes[1].bar_label(c, fmt='%.1f%%', padding=3, fontsize=9)

plt.tight_layout()
plt.show()

# Print Tables
print("is_AG_required Rates:")
ag_rate['model'] = pd.Categorical(ag_rate['model'], categories=model_order, ordered=True)
display(ag_rate.sort_values('model'))

print("\nIF Category Distribution (%):")
pivot_dist = cat_counts.pivot(index='model', columns='IF_cat', values='percentage')
pivot_dist = pivot_dist.reindex(model_order)
display(pivot_dist)